#### Middleware

Middleware란, llm이 특정 작업을 처리하기 전/후로 개입할 수 있는 기능을 말합니다.

예를 들면, 대화가 너무 길어지면 자동으로 지난 대화를 요약한다거나, 위험할 수 있는 작업을 수행하기 전에 사람의 허락을 받게 한다던가 (Human In The Loop) 하는 것이죠.

자주 사용되는 기능들은 LangChain에서 이미 구현해 둔 middleware들도 존재하고, 직접 커스텀한 middleware도 사용 가능합니다.

(이미 구현되어 있는 middleware 리스트) : https://docs.langchain.com/oss/python/langchain/middleware/built-in

In [ ]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentState, create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

credential_key=os.getenv("credential_key")
send_system_name=os.getenv("send_system_name")
model=os.getenv("model")
api_base_url=os.getenv("api_base_url")
user_id=os.getenv("user_id")

os.environ["OPENAI_API_KEY"] = 'api_key'

model = ChatOpenAI(
    model=model,
    base_url=api_base_url,
    default_headers={
        'x-dep-ticket': credential_key,
        'Send-System-Name': send_system_name,
        'User-Id': user_id,
        'User-Type': "AD_ID",
        'Prompt-Msg-Id': str(uuid.uuid4()),
        'Completion-Msg-Id': str(uuid.uuid4())
    },
    temperature=0.7,
)

c:\Users\rando\anaconda3\envs\langchain\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


본격적으로 시작하기 전에 이전 시간에 배웠던 것들을 한번씩 복습해 보겠습니다! 이전 코드를 보고 하나씩 작성해 봅시다!

1. llm이 사용할 도구들 정의하기 : [get_weather, calculator, get_today_date, save_user_info, load_user_info]

In [7]:
from langchain.tools import tool
from typing import Union
from datetime import date

@tool
def get_today_date() -> str:
    """오늘의 날짜를 YYYY-MM-DD 형식으로 반환합니다."""
    return date.today().isoformat()

@tool
def get_weather(location: str) -> str: # 문자열(str)을 받아서 문자열을 반환한다는 의미.
    """특정 장소의 날씨 정보를 알려주는 기능""" # LLM이 이 설명을 읽고 도구를 사용합니다.
    return f"It's sunny in {location}"

@tool
def calculator(a: int, b: int, operator: str) -> Union[float, str]:
    """
    2개의 정수 a, b를 입력 받으면 연산자(operator) string에
    해당하는 연산을 수행하고 그 결과를 반환하는 함수.
    operator에는 'plus', 'minus', 'multiply', 'divide' 4가지만 들어갈 수 있다.
    """
    if operator == "divide":
        if b == 0:
            # 에러를 발생시키지 않고 에이전트가 읽을 수 있는 피드백을 전달합니다.
            return "Error: 0으로 나눌 수 없습니다. b의 값을 0이 아닌 다른 숫자로 변경하세요."
        return float(a / b)

    ops = {
        "plus": a + b,
        "minus": a - b,
        "multiply": a * b,
    }
    
    if operator not in ops:
        # 지원하지 않는 연산자에 대한 가이드를 문자열로 반환합니다.
        return f"Error: '{operator}'는 지원하지 않는 연산자입니다. 'plus', 'minus', 'multiply', 'divide' 중에서 선택하세요."
    
    return float(ops[operator])

@tool
def save_user_info(info: str) -> str:
    """사용자에 대한 정보를 기록합니다. 추후 대화에 도움이 될 만한 사용자 정보를 기록하는데 사용하세요."""
    
    with open(f"ai_memory.txt", "w") as f:
        f.write(info)
    
    return "사용자 정보를 저장했습니다."

@tool
def load_user_info() -> str:
    """저장된 사용자 정보를 불러옵니다."""
    
    try:
        with open(f"ai_memory.txt", "r") as f:
            return f.read()
    except FileNotFoundError:
        return "저장된 정보가 없습니다."

2. agents.md 불러와서 `system_prompt`에 저장하기

In [3]:
with open("agents.md", encoding="utf-8") as f:
    system_prompt = f.read()

3. InmemorySaver() 메모리 추가하기

In [4]:
memory = InMemorySaver()

5. 위에서 구현한 내용들을 모두 포함해서 create_agent()로 에이전트 만들기

In [6]:
agent = create_agent(
    model,
    tools=[calculator],
    system_prompt=system_prompt,
    checkpointer=memory
)

수고하셨습니다! 이렇게 보니 양이 생각보다 적죠?

그럼 이제 본격적으로 다양한 종류의 middleware들에 대해서 알아 보도록 하겠습니다!

#### @before, after agent

agent가 실행되기 전과 후에 실행되는 middleware를 정의할 수 있습니다.

In [24]:
from langchain.agents.middleware import before_agent, after_agent
from langchain_core.messages import AIMessage
from langgraph.runtime import Runtime
from typing import Any


@before_agent(can_jump_to=["end"])
def check_message_limit(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    if len(state["messages"]) >= 3: # 에이전트가 3회 이상 호출되면 한도에 도달했다는 경고를 출력하며 에이전트가 종료됨.
        return {
            "messages": [AIMessage("Conversation limit reached.")],
            "jump_to": "end"
        }
    return None

@after_agent
def log_response(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"Model returned: {state['messages'][-1].content}")
    return None

In [22]:
agent = create_agent(
    model,
    tools=[calculator],
    system_prompt=system_prompt,
    middleware=[check_message_limit, log_response],
    checkpointer=InMemorySaver()
)

아래 코드를 3회 이상 실행해 봅시다.

In [25]:
for i in range(4):
    response = agent.invoke(
        {"messages": {"role": "user", "content": "1+1은?"}},
        {"configurable": {"thread_id": "1"}}
    )
    print(response['messages'][-1].content)

Model returned: Conversation limit reached.
Conversation limit reached.
Model returned: Conversation limit reached.
Conversation limit reached.
Model returned: Conversation limit reached.
Conversation limit reached.
Model returned: Conversation limit reached.
Conversation limit reached.


#### @wrap_tool_call

도구를 호출할 때 붙일 수 있는 미들웨어입니다!

`@wrap_tool_call`을 함수 위에 추가해주면, 도구가 호출 될 때마다 사용되는 middleware를 만들 수 있습니다!

이 미들웨어는 반드시 `request`와 `handler` 2개의 인자를 입력 받아야 합니다!

* `request` : 도구 호출 정보(도구 이름, 인자, id)
* `handler` : 실제 도구 실행 함수

이를 이용해서 도구가 호출될 때마다 로그를 작성하는 미들웨어를 생성해 보겠습니다.

In [8]:
from langchain.agents.middleware import wrap_tool_call


@wrap_tool_call
def logging_middleware(request, handler): # request와 handler 2개의 인자를 반드시 입력 받아야 합니다!
    # request : 도구 호출 정보
    # handler : 실제 도구 실행 함수
    tool_name = request.tool_call["name"] # tool 이름
    tool_args = request.tool_call["args"] # tool의 입력 인자
    print(f"[호출] 도구: {tool_name} | 인수: {tool_args}")
    try:
        result = handler(request)
        print(f"[성공] 결과: {result.content}")
        return result
    except Exception as e:
        print(f"[실패] 오류: {e}")
        raise

위 미들웨어를 agent에 추가하고 calculator를 호출하는 명령어를 입력해 봅시다!

In [10]:
agent = create_agent(
    model,
    tools=[calculator],
    middleware=[logging_middleware],
    checkpointer=InMemorySaver()
)

response = agent.invoke(
    {
        "messages": [{"role": "user", "content": "3172 x 413이 뭐야?"}],
        "user_name": "민희",
        "prefer_language": "Korean"
    },
    {"configurable": {"thread_id": "1"}}
)

print("답변 :", response['messages'][-1].content)

[호출] 도구: calculator | 인수: {'a': 3172, 'b': 413, 'operator': 'multiply'}
[성공] 결과: 1310036.0
답변 : 3172 × 413 = **1,310,036** 입니다.


#### 파일 탐색 Middleware

이 middleware는 특이하게 에이전트에게 파일을 찾을 수 있는 권한을 부여해 줍니다.

현재 폴더에 있는 `emails.txt`의 내용을 읽고, '프로젝트 킥오프 미팅 일정'에 대해서 물어보도록 하겠습니다.

In [ ]:
from langchain.agents.middleware import FilesystemFileSearchMiddleware

# 이 에이전트가 볼 수 있는 경로를 현재 경로로 제한함.
fs_middleware = FilesystemFileSearchMiddleware(root_path=".")

In [27]:
mail_agent = create_agent(
    model,
    tools=[],
    system_prompt="당신은 유용한 AI 어시스턴트입니다. 파일을 찾거나 내용을 검색할 때 glob_search와 grep_search 도구를 활용하세요.",
    middleware=[fs_middleware, logging_middleware]
)

response = mail_agent.invoke(
    {"messages": [{"role": "user", "content": "프로젝트 킥오프 미팅 일정이 언젠지 알려줘."}]}
)

print(response['messages'][-1].content)

[호출] 도구: glob_search | 인수: {'pattern': '**/*', 'path': '/'}
[호출] 도구: grep_search | 인수: {'pattern': '프로젝트 킥오프|킥오프|kickoff|kick-off|project kickoff', 'path': '/', 'include': '*', 'output_mode': 'content'}
[성공] 결과: /00_LangChain_LCEL.ipynb
/01_LangChain_basic-정답.ipynb
/01_LangChain_basic.ipynb
/02_LangChain_Middleware-정답.ipynb
/02_LangChain_Middleware.ipynb
/03_LangChain_monitoring.ipynb
/agents.md
/emails.txt
/basic_agent\agent.py
/basic_agent\agents.md
/basic_agent\agent_notebook.ipynb
/basic_agent\basic_agent.png
/basic_agent\main.py
/basic_agent\middleware.py
/basic_agent\tools.py
/basic_agent-DIY\agent_notebook.ipynb
/basic_agent-DIY\basic_agent.png
/basic_agent-DIY\main.py
/basic_agent-DIY\agent\agent.md
/basic_agent-DIY\agent\agent.py
/basic_agent-DIY\agent\middleware.py
/basic_agent-DIY\agent\tools.py
[성공] 결과: /02_LangChain_Middleware-정답.ipynb:394:    "현재 폴더에 있는 `emails.txt`의 내용을 읽고, '프로젝트 킥오프 미팅 일정'에 대해서 물어보도록 하겠습니다."
/02_LangChain_Middleware-정답.ipynb:424:    "    {\"messages\": 

#### Human In The Loop

파일 작업, 메일 전송 등 민감한 작업을 할 때 꼭 인간의 허락을 받고 실행하도록 하는 미들웨어입니다.

In [ ]:
from langchain.tools import tool
from langgraph.types import Command
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

@tool
def read_email_tool(email_id: str) -> str:
    """email_id를 제공하면 해당 메일의 내용을 반환하는 툴"""
    return f"Email content for ID: {email_id}"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """수신자, 제목, 내용을 입력 받으면 실제로 메일을 발송하는 툴"""
    return f"Email sent to {recipient}"

# 에이전트 생성
agent = create_agent(
    model,
    tools=[read_email_tool, send_email_tool], 
    checkpointer=InMemorySaver(),
    system_prompt="당신은 메일 비서입니다. 메일을 보낼 때는 반드시 'send_email_tool' 도구를 호출해야 합니다. 절대로 말로만 보냈다고 하지 마세요.",
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                }
            }
        ),
    ],
)

response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "삼성이라는 이름으로 test@gmail.com으로 생일 파티 초대 메일 작성한 다음 보내줘"}
        ]
    },
    {"configurable": {"thread_id": "1"}}
)



# 결과 확인
if "__interrupt__" in response:
    print("메일을 작성하기 전에 허락을 해주셔야 합니다.")

# 1. '승인' 명령어를 담은 객체를 만듭니다.
approval_command = Command(resume={"decisions": [{"type": "approve"}]})

# 2. 에이전트에게 다시 invoke를 날립니다. 
final_response = agent.invoke(
    approval_command,
    {"configurable": {"thread_id": "1"}}
)

# 3. 드디어 에이전트가 툴을 실행하고 내놓은 최종 결과를 확인합니다.
print("에이전트의 최종 답변:")
print(final_response['messages'][-1].content)

메일을 작성하기 전에 허락을 해주셔야 합니다.
🤖 에이전트의 최종 답변:
메일을 보냈습니다.


##### SummarizationMiddleware

대화가 길어지게 되면 이전 대화 내용들의 토큰 수가 점점 부담될 수 있습니다.

이럴 땐 `SummarizationMiddleware`를 활용해 이전의 대화 기록을 요약할 수 있습니다!

* model : 요약을 수행할 모델의 이름.
* trigger : 요약을 진행할 조건.
    * tokens : 토큰 수를 기준으로
    * messages : 메세지 수를 기준으로
    * fraction : 모델 최대 컨텍스트 크기 대비 비율 (0~1.0)
* keep : 요약 후 얼마나 남길 지.
    * messages : 최근 n개 메세지만 유지
    * tokens : 최근 n개 토큰만 유지

요약 확인을 위해 `trigger`와 `keep`을 작게 설정해 보겠습니다.

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

text_summarizer = SummarizationMiddleware(
    model=model,
    base_url=api_base_url,
    default_headers={
        'x-dep-ticket': credential_key,
        'Send-System-Name': send_system_name,
        'User-Id': user_id,
        'User-Type': "AD_ID",
        'Prompt-Msg-Id': str(uuid.uuid4()),
        'Completion-Msg-Id': str(uuid.uuid4())
    },
    trigger=("tokens", 1000),
    keep=("messages", 2)
)

agent = create_agent(
    model,
    middleware=[
        text_summarizer
    ],
    checkpointer=InMemorySaver()
)

이제 200토큰이 넘어갈 때까지 대화를 한 뒤...

In [39]:
response = agent.invoke(
    {
        "messages": {"role": "user", "content": "일 하기 싫은데 나한테 문제가 있는걸까?"},
        "user_name": "minhee0219",
        "prefer_language": "Korean"
    },
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

response = agent.invoke(
    {
        "messages": {"role": "user", "content": "일하지 않고 돈을 벌고 싶어..."},
        "user_name": "minhee0219",
        "prefer_language": "Korean"
    },
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

response = agent.invoke(
    {
        "messages": {"role": "user", "content": "하루종일 자고 싶기도 해."},
        "user_name": "minhee0219",
        "prefer_language": "Korean"
    },
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

그럴 수도 있고, 아닐 수도 있어요.  
**“일 하기 싫다” = 문제가 있다** 는 뜻은 아니에요. 아주 흔한 감정이에요.

대충 나눠보면 이런 경우가 많아요:

- **그냥 지친 경우**: 쉬어야 하는데 계속 버티고 있을 때
- **일이 너무 안 맞는 경우**: 흥미가 없거나 의미가 안 느껴질 때
- **스트레스/불안이 큰 경우**: 시작 자체가 부담스러워서 회피하게 될 때
- **번아웃**: 의욕이 바닥나고 아무것도 하기 싫어질 때
- **우울감**: 일뿐 아니라 전반적으로 무기력할 때

한 번 이렇게 체크해보면 좋아요:
1. **일만 하기 싫은지**, 아니면 **전반적으로 다 하기 싫은지**
2. **쉬면 좀 나아지는지**, 아니면 **쉬어도 계속 힘든지**
3. **싫은 이유가 명확한지**  
   - 일이 너무 많음  
   - 사람 때문  
   - 성취감 없음  
   - 미래가 불안함  
   - 그냥 몸이 너무 피곤함

만약
- 2주 이상 계속 무기력하고,
- 잠/식욕/집중이 많이 망가지고,
- 예전엔 괜찮던 것도 다 버겁다면

그건 “게으름”보다 **번아웃이나 우울 상태**일 수 있어서 도움을 받는 게 좋아요.

지금은 스스로를 너무 몰아붙이기보다,  
**“내가 게으른가?”** 보다는  
**“내가 지금 어떤 상태라서 싫은가?”** 로 보는 게 더 정확해요.

원하면 제가 질문 몇 개만 해서  
**번아웃인지, 그냥 일 싫은 건지, 우울감에 가까운지** 같이 구분해드릴게요.
그 마음, 정말 자연스러워요.  
**“일은 싫은데 돈은 필요하다”**는 건 사실 많은 사람이 갖는 생각이에요. 이상한 게 아니에요.

다만 현실적으로는, **완전히 아무 일도 하지 않고 돈이 들어오게 만드는 방법은 거의 없고**, 보통은  
- 처음에 한 번 열심히 만들거나,
- 돈이 굴러가게 만드는 구조를 만들거나,
- 아주 적은 노동으로 유지되는 방식  
중 하나예요.

예를 들면:
- **자산에서 나오는 수익**: 예금 이자, 배당, 임대료, 투자 수익
- **

결과를 확인해 볼까요?

In [40]:
state = agent.get_state({"configurable": {"thread_id": "1"}}).values

for message in state['messages']:
    print(message.content)

Here is a summary of the conversation to date:

## SESSION INTENT
사용자는 일을 하고 싶지 않다는 감정이 이상한 것인지 묻는 상태에서, 이어서 “일하지 않고 돈을 벌고 싶다”고 말하며 자신의 욕구와 상태를 이해받고 싶어함. 현재 목표는 이 감정을 비난 없이 다루고, 번아웃/우울/스트레스/적성 문제와 같은 가능성을 구분하며 대화를 이어가는 것.

## SUMMARY
- 사용자 최초 질문: “일하기 싫다는데 내가 이상한 걸까?”라는 취지의 한국어 메시지.
- 이에 대해 “일하기 싫다 = 문제가 있다”는 뜻은 아니며, 흔한 감정이라고 정상화하는 답변이 제공됨.
- 가능한 원인으로 다음이 제시됨:
  - 단순 피로/휴식 필요
  - 일이 맞지 않음, 흥미 부족
  - 스트레스/불안으로 시작 자체가 부담
  - 번아웃
  - 우울감/전반적 무기력
- 자가 점검 질문도 제안됨:
  1. 일만 하기 싫은지, 전반적으로 모든 것이 하기 싫은지
  2. 쉬면 나아지는지, 쉬어도 계속 힘든지
  3. 싫은 이유가 명확한지(업무량, 사람, 성취감 부족, 미래 불안, 신체 피로 등)
- 2주 이상 지속되는 무기력, 수면/식욕/집중의 저하, 예전엔 괜찮던 것도 버거운 경우는 게으름보다 번아웃이나 우울 상태일 수 있다고 언급함.
- 마지막 제안: 질문 몇 개를 통해 번아웃인지, 단순히 일이 싫은 건지, 우울감에 가까운지 구분해줄 수 있다고 했음.
- 이후 사용자 추가 메시지: “일하지 않고 돈을 벌고 싶어...”  
  이는 노동 회피/경제적 자유에 대한 욕구를 더 직접적으로 드러내는 발언이며, 아직 이에 대한 후속 응답은 없음.

## ARTIFACTS
None

## NEXT STEPS
- 사용자의 “일하지 않고 돈을 벌고 싶어”라는 욕구를 비난 없이 공감적으로 받아들이기.
- 현실적으로 가능한 방향인지와 그 욕구의 배경(번아웃, 노동 스트레스, 경제적 압박, 자유 욕구 등)을 짧게 탐색하기.
- 필요하면 “완전 무노동 수입”의 

#### 컨텍스트 자동 관리

앞서 SummarizationMiddleware를 이용해 대화기록을 자동으로 요약하는 작업을 수행해 봤습니다.

이번엔 ContextEditingMiddleware를 소개하겠습니다! ContextEditingMiddleware도 대화가 길어질 경우 컨텍스트를 관리하기 위한 미들웨어입니다.

SummarizationMiddleware와의 차이점은...

| 특성 | SummarizationMiddleware | ContextEditingMiddleware |
|------|------------------------|--------------------------|
| 방식 | 인공지능(LLM)이 대화 내용을 요약함 | 불필요한 정보(도구 결과 등)를 직접 삭제함 |
| 비용 | 요약을 위해 인공지능을 추가로 써야 해서 비용 발생 | 직접 지우는 방식이라 추가 비용 없음 |
| 정보 보존 | 핵심 내용을 요약해서 남기므로 의미가 유지됨 | 선택한 정보를 완전히 지우므로 해당 정보는 사라짐 |
| 적용 대상 | 모든 대화 메세지 | 주로 도구가 내놓은 결과물(결과 값 등) |
| 사용 시나리오 | 대화가 너무 길어져서 맥락 유지가 힘들 때 | 일시적인 데이터나 너무 큰 도구 결과가 부담될 때 |

따라서 비용이 들지 않는 ContextEditingMiddleware로 먼저 불필요한 도구 사용 기록을 제거하고, 그 뒤에도 컨텍스트가 부족하다면 SummarizationMiddleware를 사용하는 식으로 구성하는 것이 일반적입니다!

**어떻게 미들웨어의 순서를 컨트롤하나요?**

create_agent()의 middleware 리스트 인자에 들어가는 순서가 middleware를 사용하는 순서가 됩니다.

`middleware=[ContextEditingMiddleware, SummarizationMiddleware]` : ContextEditing을 먼저 하고, Summarization을 수행함.

In [38]:
from langchain.agents.middleware import SummarizationMiddleware, ContextEditingMiddleware, ClearToolUsesEdit

agent = create_agent(
    model,
    tools=[wikipedia_search, calculator],
    system_prompt="당신은 유용한 AI 어시스턴트입니다. 도구를 사용할 수 있는 경우 꼭 도구를 사용해서 정확한 결과를 답변해야 합니다.",
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(
                    trigger=200, # 토큰 수가 200이 넘어가면 실행됨.
                    keep=2, # 가장 최근 2개의 도구 결과는 남김.
                    exclude_tools=["calculator"] # calculator 도구 실행 결과는 지우지 않는다.
                )
            ]
        ),
        SummarizationMiddleware( # 만약 앞서 ContextEditingMiddleware를 수행하고도 토큰 수가 1000이 넘어간다면 수행된다.
            model=model_name,
            trigger=("tokens", 1000),
            keep=("messages", 2)
        )
    ],
    checkpointer=InMemorySaver()
)

In [39]:
messages = [
    "조선시대 왕 영조의 업적에 대해서 알려줘.",
    "358 x 13은?",
    "지금 내 메일 기록(emails.txt) 중에 '영희'라는 사람한테 온 메일이 있어?",
    "코뿔소의 몸무게는 보통 어떻게 돼?",
    "메일 내용 중에 일정 관련된게 있나?"
]

In [42]:
for message in messages:
    response = agent.invoke(
        {
            "messages": [{"role": "user", "content": message}]
        },
        {"configurable": {"thread_id": "1"}}
    )

    print("답변 :", response['messages'][-1].content)

답변 : 조선시대 **영조**의 업적은 쉽게 말해 **나라를 안정시키고 백성의 부담을 줄이려 한 것**이에요.

## 영조의 주요 업적

### 1. 탕평책 실시
- 당시 조선은 **붕당 싸움**이 심했어요.
- 영조는 어느 한쪽 편만 들지 않고 **여러 당파의 인재를 골고루 등용**했어요.
- 이 정책을 **탕평책**이라고 해요.
- 덕분에 정치적 갈등이 조금 완화되고 왕권이 강화되었어요.

### 2. 균역법 시행
- 백성들은 군대에 갈 대신 **군포**라는 세금을 냈어요.
- 원래는 **2필**을 내야 했는데, 영조는 이를 **1필**로 줄였어요.
- 그래서 백성들의 부담이 크게 줄었어요.
- 부족한 세금은 다른 방법으로 보충했어요.

### 3. 법과 제도 정비
- 나라 운영의 기준을 다시 정리했어요.
- 대표적으로 **속대전** 편찬을 추진해서 행정 질서를 바로잡았어요.

### 4. 왕권 강화
- 붕당 싸움을 줄이고 왕이 중심이 되는 정치를 하려고 했어요.
- 이를 통해 조선의 국정을 더 안정적으로 운영하려 했어요.

### 5. 사회 안정 노력
- **이인좌의 난** 같은 반란을 진압하며 나라의 질서를 지켰어요.
- 큰 혼란 없이 나라를 다스리기 위해 노력했어요.

### 6. 교육과 문화 중시
- 성균관, 향교 같은 교육 기관을 유지하고 유교 질서를 강조했어요.
- 학문과 예절을 중요하게 여겼어요.

---

## 한 줄 요약
**영조는 탕평책과 균역법을 통해 정치적 갈등을 줄이고 백성의 부담을 덜어준 왕이다.**

원하시면 제가 이것을 **시험답안처럼 3줄로 요약**해드릴게요.
답변 : 358 × 13 = **4654**입니다.
답변 : 현재는 제가 `emails.txt` 파일 내용을 직접 볼 수 없어서, **영희에게 온 메일이 있는지 확인할 수 없습니다.**

`emails.txt` 내용을 여기 붙여주시면 제가 바로 찾아드릴게요.  
원하시면 제가 **“영희”가 보낸 메일인지 / “영희”에게 온 메일인지** 둘 다 구분해서 확인해드릴 수 있습

In [43]:
state = agent.get_state({"configurable": {"thread_id": "1"}}).values

for message in state['messages']:
    print(message.content)

Here is a summary of the conversation to date:

## SESSION INTENT
사용자는 `emails.txt`의 메일 내용 중 **일정 관련 정보가 있는지 확인**하려고 함. 이전 맥락상 ‘영희’에게 온 메일 존재 여부 확인 요청이 있었고, 현재는 메일 내용에서 일정 관련 문구가 있는지 추가로 확인하려는 상황임.

## SUMMARY
- 이전 대화에서는 조선 제21대 왕 **영조의 업적**을 한국어로 쉽게 설명하는 작업이 있었음.
- 영조 관련 핵심 정리:
  - 영조(1694~1776), 본명 **이금**, 즉위 전 **연잉군**
  - 숙종의 둘째 아들이며 후궁 **숙빈 최씨** 소생
  - 재위: **1724~1776**
  - 주요 업적:
    1. **탕평책**으로 붕당 갈등 완화 및 인재 등용
    2. **균역법(1750)** 시행으로 군포 부담 완화
    3. **속대전** 편찬 추진 등 법·제도 정비
    4. 왕권 강화 및 정치 안정
    5. 교육·문화 진흥
    6. **이인좌의 난(1728)** 등 반란 대응으로 사회 안정
- 시험용 짧은 요약도 제공됨:
  - “영조는 탕평책과 균역법을 통해 정치적 갈등을 줄이고 백성의 부담을 덜어준 왕이다.”
- 이후 사용자가 계산 질문 `358 x 13은?`을 했고, 답은 **4654**였음.
- `emails.txt`에서 ‘영희’ 관련 메일 존재 여부를 확인하려 했으나, 당시에는 **파일 내용을 직접 볼 수 없어 확인 불가**하다고 안내했음.
- 이후 사용자가 **코뿔소의 몸무게**를 물었고, 위키피디아 검색을 통해 여러 종의 몸무게 정보를 확인 중이었음.
  - 검색 쿼리: `Rhino average weight`, `White rhinoceros weight`, `Black rhinoceros weight`, `Indian rhinoceros weight`
  - 반환 요약:
    - **White rhinoceros**: 가장 큰 현생 코뿔